# Phase 2 · Week 5 — Model Comparison: Logistic vs. Tree Ensembles

**Question:** Does a more *expressive* model class find predictive edge where logistic regression didn't? This compares three model families — **linear** (logistic regression), **bagging** (Random Forest), and **boosting** (XGBoost) — on the *same* 18-feature next-day-direction task, with rigorous out-of-sample evaluation.

**Why it matters:** if a flexible, nonlinear ensemble *also* finds nothing out of sample, the "no edge" result can't be blamed on the model — the ceiling is the **problem/features**, not the algorithm. That closes the "maybe I just needed a better model" objection.

**Setup:**
- Same 18-feature dataset (see `FEATURES.md`), pooled across the 7 tickers, chronological 60/20/20 split.
- Logistic is **scaled** (`StandardScaler`, fit on train only); trees are **unscaled** (they split on thresholds, so they're scale-invariant).
- Headline metric: **CV accuracy vs. the base rate** (the majority-class "always predict up" benchmark). Beating 50% means nothing; beating the *base rate* is the honest bar.

In [1]:
import sys
sys.path.append("src")

import pandas as pd
from load import data_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

x_train, x_cv, x_test, y_train, y_cv, y_test = data_split()
base_rate = y_cv.mean()   # majority-class 'always up' benchmark on the CV set

# Logistic regression (needs scaling)
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_cv_s = scaler.transform(x_cv)
logreg = LogisticRegression(max_iter=1000, random_state=55).fit(x_train_s, y_train)
acc_log = accuracy_score(y_cv, logreg.predict(x_cv_s))

# Random Forest (no scaling)
rf = RandomForestClassifier(random_state=55).fit(x_train, y_train)
acc_rf = accuracy_score(y_cv, rf.predict(x_cv))

# XGBoost (no scaling)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=55, verbosity=0).fit(x_train, y_train)
acc_xgb = accuracy_score(y_cv, xgb.predict(x_cv))

comparison = pd.DataFrame({
    "Model": ["Base rate (always up)", "Logistic Regression", "Random Forest", "XGBoost"],
    "Family": ["\u2014", "Linear", "Bagging", "Boosting"],
    "CV accuracy": [base_rate, acc_log, acc_rf, acc_xgb],
})
comparison["Edge over base"] = comparison["CV accuracy"] - base_rate
comparison.round(4)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,Model,Family,CV accuracy,Edge over base
0,Base rate (always up),—,0.5215,0.0000
1,Logistic Regression,Linear,0.5268,0.0053
2,Random Forest,Bagging,0.5075,-0.0140
3,XGBoost,Boosting,0.5162,-0.0053


## Headline

All three model families land **at the base rate** — the edge over "always predict up" is a rounding error (sub-1%) for every one of them. A linear model, a bagged forest, and a boosted ensemble all agree: no meaningful out-of-sample signal in these features.

The sections below show the deeper diagnostics behind each model, plus the walk-forward robustness check, the SHAP interpretability layer, and the class-weight test.

## 1. Logistic regression (linear baseline)

*(`src/model.py`)*

On the 18-feature set, CV accuracy is ~**0.527** vs a base rate of ~**0.521** — about **0.6 points** of edge, i.e. noise. A polynomial-degree sweep makes it worse, not better: training accuracy climbs toward ~0.71 while CV accuracy *drops* — pure overfitting. Adding capacity never lifts CV above the base rate.

## 2. Random Forest (bagging)

*(`src/random_forest.py`)*

Hyperparameter sweeps over `min_samples_split`, `max_depth`, and `n_estimators` all tell the same story: **training accuracy swings up to 1.0** (deep, unconstrained trees memorize every row) while **validation accuracy stays pinned at ~0.50–0.53** — the base rate — at *every* setting. No configuration lets a bagged, nonlinear ensemble clear the baseline. Constraining the trees only closes the train–CV gap by pulling *train down*, never by pulling *CV up*.

## 3. XGBoost (boosting)

*(`src/xgb_model.py`)*

The learning curve (train vs CV logloss over boosting rounds) is the clearest result: **train logloss falls steadily** while **CV logloss sits at ~0.693** and drifts *up*. That value is `ln(2)` — the exact logloss of predicting 0.5 for everything, i.e. the *no-information* level. Early stopping picks **round ~1 as best**: given 500 trees, the model concludes the best thing to do is add essentially none. Competition-winning boosting, and it finds nothing to boost.

## 4. Walk-forward validation (robustness across time)

*(`src/walk_forward.py`)*

The strongest test: instead of one fixed split, roll an 18-month train / 6-month test window forward through history and evaluate on each. This asks whether the result is *robust* or just an artifact of one period.

Across the windows, XGBoost accuracy hovers **around each window's base rate** — above it in some windows, below in others, averaging slightly *below*. Accuracy stays flat (~0.50–0.54) while the base rate itself swings from ~0.47 to ~0.59, meaning the model isn't reading the regime — it's guessing near-randomly and occasionally matching the period's tilt by luck.

So the no-edge result holds across independent time periods — it is **not** a one-split fluke.

## 5. SHAP — what the model leaned on

*(`src/xgb_model.py`, `run_shap_plot`)*

SHAP attributes each prediction to its features. The XGBoost model concentrated on **Return, RSI, MACD histogram, lagged returns, Volume Ratio, and Bollinger position** — the price/momentum features. The **calendar dummies were negligible** (as expected — calendar effects are weak), and **Log Return ranked ~0** (redundant with Return, correctly ignored). Both are reassuring sanity checks that the model looked in sensible places.

**But** the beeswarm shows *no clean directional relationship* for any feature: high and low feature values are smeared across *both* sides of zero, and the magnitudes are tiny. If a feature truly predicted direction, its colors would separate cleanly. They don't. So high SHAP importance here means **"the model leaned on this to fit the training data,"** not **"this predicts the future"** — exactly consistent with the zero out-of-sample edge.

## 6. Class weights (imbalance check)

*(`src/xgb_model.py`)*

The target is mildly imbalanced (~54% up / 46% down). Applying class weighting (`scale_pos_weight ≈ 0.85`) to counter it had **no effect** on out-of-sample performance — the CV logloss curve stayed at ~0.693, unchanged from the unweighted model. The imbalance was never the limiting factor.

## Conclusion

Across the full spectrum of tabular ML — **linear (logistic), bagging (Random Forest), and boosting (XGBoost)** — and under the most rigorous validation available (**walk-forward across time**), **no model achieves out-of-sample edge** on next-day direction. Adding model capacity only ever increased *overfitting*, never *generalization*; SHAP shows no feature carries a clean directional signal; and class weighting made no difference.

The consistent verdict across every model family and validation scheme means the ceiling is the **problem/features**, not the algorithm. **Next-day direction on liquid equities and crypto, from price/volume-derived features, is effectively unpredictable** — a robust, honest negative result. This closes the "maybe a better model would work" objection: the better models were tried, and they agree.

*(Tree-based note: LightGBM was skipped — it is XGBoost's close cousin (gradient-boosted trees) and would reproduce the XGBoost result. Documented here rather than re-run.)*

## Reproduce

| Analysis | Script |
|---|---|
| Feature pipeline + lookahead audit | `src/features.py`, `FEATURES.md` |
| Logistic baseline + degree sweep | `src/model.py` |
| Random Forest hyperparameter sweeps | `src/random_forest.py` |
| XGBoost learning curve, SHAP, class weights | `src/xgb_model.py` |
| Walk-forward validation | `src/walk_forward.py` |